# Week 4 - Word embedding untuk klasifikasi SMS spam
Marvel Kevin Nathanael | 00000108042

Mengikuti TLM 4: tokenisasi, embedding yang dilatih dari data, lalu klasifikasi. Dataset tersedia lokal di folder data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score

tf.keras.utils.set_random_seed(42)
path = Path("data/SMSSpamCollection")
if not path.exists():
    path = Path("Week 4/data/SMSSpamCollection")
df = pd.read_csv(path, sep="	", names=["label", "text"], encoding="latin-1")
df = df.drop_duplicates(subset="text").reset_index(drop=True)
df["target"] = (df["label"] == "spam").astype("int32")
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["target"], random_state=42)
train_df, val_df = train_test_split(
    train_df, test_size=0.2, stratify=train_df["target"], random_state=42)
print("Train/val/test:", len(train_df), len(val_df), len(test_df))
print("Proporsi spam train:", round(train_df["target"].mean(), 3))

In [ ]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=10000, output_mode="int", output_sequence_length=60)
vectorizer.adapt(train_df["text"].to_numpy())  # Vocab hanya dari train.
vocab_size = len(vectorizer.get_vocabulary())
model = tf.keras.Sequential([
    tf.keras.Input(shape=(), dtype=tf.string),
    vectorizer,
    tf.keras.layers.Embedding(vocab_size, 64, mask_zero=True),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(curve="PR", name="pr_auc")])
counts = train_df["target"].value_counts()
class_weight = {int(k): len(train_df) / (2 * int(v)) for k, v in counts.items()}
history = model.fit(
    train_df["text"].to_numpy(), train_df["target"].to_numpy(),
    validation_data=(val_df["text"].to_numpy(), val_df["target"].to_numpy()),
    epochs=15, batch_size=64, class_weight=class_weight, verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_pr_auc", mode="max", patience=3, restore_best_weights=True)])

In [ ]:
prob = model.predict(test_df["text"].to_numpy(), verbose=0).ravel()
pred = (prob >= 0.5).astype(int)
print(classification_report(test_df["target"], pred,
                            target_names=["ham", "spam"], digits=3))
print("PR-AUC test:", round(average_precision_score(test_df["target"], prob), 4))
print("Confusion matrix:\n", confusion_matrix(test_df["target"], pred))
print("Baseline majority accuracy:", round(1 - test_df["target"].mean(), 4))

**Catatan:** Teks duplikat dibuang sebelum split agar SMS yang sama tidak masuk train dan test. Embedding dilatih dari awal, belum GloVe pretrained. Dataset berbahasa Inggris dan relatif kecil; hasil belum tentu berlaku untuk SMS bahasa Indonesia. PR-AUC dan recall spam perlu dibaca bersama akurasi.